# Controls after the first baseline

1. **E1b - leakage attribution:** the official split with its leakage removed (train/val de-duplicated), SAME 64 official test images.
2. **Noise floor, fixed-epoch protocol:** clean folds 0 and 1, seeds 42/43/44, 100 epochs, no early stopping, primary metric from `last.pt`
   (EMA weights, no validation-based selection); `best.pt` is also scored so the two protocols can be compared.

Settings: Accelerator *GPU T4 x2*, Internet ON. First run with `SMOKE = True` (2 epochs, 2 runs) in an interactive session, then set `SMOKE = False` and *Save Version -> Save & Run All*.


In [ ]:
import os, sys, subprocess
REPO_URL = "https://github.com/nithoukhan1/hbfmid.git"
REPO = "/kaggle/working/hbfmid"
if os.path.exists(REPO):
    subprocess.run(["git", "-C", REPO, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics==8.4.215"], check=True)
os.environ["WANDB_MODE"] = "disabled"
import ultralytics, torch
print("ultralytics", ultralytics.__version__, "| torch", torch.__version__)
print("GPUs:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
assert torch.cuda.is_available(), "No GPU: set Accelerator to GPU T4 x2, then restart."
print("repo commit:", subprocess.run(["git", "-C", REPO, "log", "--oneline", "-1"], capture_output=True, text=True).stdout.strip())
assert os.path.exists(f"{REPO}/data_splits/official_dedup/train.txt"), "official_dedup lists missing: push the repo first"


In [ ]:
DATA_ROOT = "/kaggle/input/datasets/nithoukhan2/hbfmid/Human Bone Fractures Multi-modal Image Dataset (HBFMID)/Bone Fractures Detection"
assert os.path.isdir(DATA_ROOT), DATA_ROOT
SMOKE = True
MODEL = "yolov8s.pt"
USE_ALL_GPUS = True
if SMOKE:
    EPOCHS, OUT = 2, "/kaggle/working/results/controls_smoke"
    PLAN = [("official_dedup", 0, 42), ("clean", 0, 42)]
else:
    EPOCHS, OUT = 100, "/kaggle/working/results/controls"
    PLAN  = [("official_dedup", 0, 42)]                                   # E1b
    PLAN += [("clean", f, s) for f in (0, 1) for s in (42, 43, 44)]       # noise floor, fixed protocol
print(len(PLAN), "runs, epochs =", EPOCHS, "->", OUT)


In [ ]:
import queue, threading, time
os.makedirs(f"{OUT}/logs", exist_ok=True)
gpus = list(range(torch.cuda.device_count())) if USE_ALL_GPUS else [0]
q = queue.Queue()
for t in PLAN:
    q.put(t)
failed = []

def worker(gpu):
    while True:
        try:
            mode, fold, seed = q.get_nowait()
        except queue.Empty:
            return
        name = f"{mode}_f{fold}_s{seed}"
        cmd = [sys.executable, f"{REPO}/scripts/training/run_baseline.py",
               "--mode", mode, "--fold", str(fold), "--seed", str(seed), "--model", MODEL,
               "--epochs", str(EPOCHS), "--fixed-epochs", "--device", str(gpu), "--data-root", DATA_ROOT,
               "--splits-dir", f"{REPO}/data_splits", "--out", OUT]
        t0 = time.time()
        with open(f"{OUT}/logs/{name}.log", "w") as f:
            rc = subprocess.run(cmd, stdout=f, stderr=subprocess.STDOUT).returncode
        print(f"[GPU {gpu}] {name}: {'OK' if rc == 0 else 'FAILED rc=' + str(rc)} in {(time.time()-t0)/60:.1f} min", flush=True)
        if rc != 0:
            failed.append(name)

threads = [threading.Thread(target=worker, args=(g,)) for g in gpus]
[t.start() for t in threads]; [t.join() for t in threads]
print("failed runs:", failed or "none")
for n in failed:
    print("\n==== tail of", n)
    print("".join(open(f"{OUT}/logs/{n}.log").readlines()[-25:]))


In [ ]:
import shutil
subprocess.run([sys.executable, f"{REPO}/scripts/evaluation/aggregate_controls.py", "--runs", OUT,
                "--e1", f"{REPO}/results/baseline/baseline_results/official_yolov8s_s42/metrics.json"])
shutil.make_archive("/kaggle/working/controls_results", "zip", OUT)
print(sorted(os.listdir(OUT)))
if SMOKE:
    print("\nSMOKE test done. Check output above, then set SMOKE = False and use Save Version.")
if failed:
    print("\nWARNING: failed runs:", failed)


After the real run: download `controls_results.zip` from the Output tab, unzip into `results/controls/` in the local repo, commit and push.
